# Unichart: `set_plot_size` + the new suptitle / legend layout

This notebook demonstrates two related features added to `unichart.py`:

1. **Smarter suptitle & legend handling** — a horizontal legend can no longer
   cover the suptitle, even when the title spans several lines and/or the legend
   wraps to many rows. The title font size is also accounted for automatically.
2. **`set_plot_size`** — pin the *inner plot area* to a fixed size so plots stay
   the same size no matter what the suptitle or legend are doing.

Everything below is self-contained (synthetic data) and safe to *Run All*.

### Dependencies

`unichart` needs `scipy` and `ipywidgets` (and `kaleido` only if you want to
export static images). If importing fails, install with:

```bash
pip install scipy ipywidgets kaleido          # or, on a managed system python:
pip install --break-system-packages scipy ipywidgets kaleido
```

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))   # so `import unichart` resolves to this repo

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import unichart as uc

np.random.seed(42)

## 0. Synthetic data + a measurement helper

`make_run` fabricates one experiment's worth of columns. We load a few runs into
a notebook for the general demos, and a *separate* notebook with many runs later
to force the legend to wrap.

In [2]:
def make_run(n=80, drift=0.0, scale=1.0):
    x = np.linspace(0, 10, n)
    return pd.DataFrame({
        'time':   x,
        'signal': np.cumsum(np.random.randn(n)) * scale + drift * x,
        'temp':   20 + 5 * np.sin(x) + np.random.randn(n),
        'power':  np.abs(np.random.randn(n)) * 10 * scale,
        'group':  np.random.choice(['A', 'B', 'C'], n),
    })

nb = uc.UnichartNotebook()
for i in range(3):
    nb.load_df(make_run(drift=0.5 * i, scale=1 + 0.3 * i), title=f'run_{i}')

UniChart Notebook Environment Initialized.
Loaded Set 0: run_0
Loaded Set 1: run_1
Loaded Set 2: run_2


In [3]:
def paper_region(fig):
    """Inner plotting area in px = figure size minus margins.

    Margins left unset fall back to Plotly's defaults (l=r=b=80, t=100). This
    equals the actual rendered plot rectangle whenever margin autoexpand stays
    inert (the normal case for titles and the downward-growing legend)."""
    L = fig.layout
    m = L.margin
    l = 80  if m.l is None else m.l
    r = 80  if m.r is None else m.r
    t = 100 if m.t is None else m.t
    b = 80  if m.b is None else m.b
    w = (L.width  or 700) - l - r
    h = (L.height or 450) - t - b
    return (round(w, 1), round(h, 1))

---
## 1. Suptitle & legend handling

### What changed

The suptitle is pinned to the top of the figure **container** and the horizontal
("above") legend is now **top-anchored just below the title**, in container
coordinates. Previously the legend was *bottom-anchored* right above the plot, so
extra legend rows grew **upward into the title**. Now they grow **downward toward
the plot**, and the top margin is sized from the number of title lines (and the
title font size) so there is always room.

### 1a. Multi-line suptitle (`<br>`) — legend sits safely below

Three variables → three subplots, each dataset a legend entry. The 3-line title
is fully clear of the legend.

In [4]:
nb.plot(
    x='time', y=['signal', 'temp', 'power'],
    suptitle=('Sensor Overview — Multi-line Title'
              '<br>Second line: three variables across three runs'
              '<br>Third line: the legend sits safely below the title'),
    figsize=(11, 6),
)

### 1b. Newlines (`\n`) work too

Plotly itself only breaks lines on `<br>`. Unichart now normalizes `\n` to `<br>`
so a Python newline both renders as a line break *and* gets counted when reserving
space.

In [5]:
nb.plot(
    x='time', y='signal',
    suptitle='Title written with newlines\nsecond line\nthird line',
    figsize=(11, 5),
)

### 1c. A wrapping legend grows *downward*, never into the title

Here we use a fresh notebook with **14 runs** so the horizontal legend wraps to
several rows. Watch how the rows stack *below* the title toward the plot.

In [6]:
nb_many = uc.UnichartNotebook()
for i in range(14):
    nb_many.load_df(make_run(drift=0.3 * i), title=f'experiment_run_{i:02d}')

nb_many.plot(
    x='time', y='signal',
    suptitle=('Wrapping Legend Demo'
              '<br>14 series → the legend wraps to several rows and grows downward,'
              '<br>never up into this title'),
    figsize=(11, 6),
)

UniChart Notebook Environment Initialized.
Loaded Set 0: experiment_run_00
Loaded Set 1: experiment_run_01
Loaded Set 2: experiment_run_02
Loaded Set 3: experiment_run_03
Loaded Set 4: experiment_run_04
Loaded Set 5: experiment_run_05
Loaded Set 6: experiment_run_06
Loaded Set 7: experiment_run_07
Loaded Set 8: experiment_run_08
Loaded Set 9: experiment_run_09
Loaded Set 10: experiment_run_10
Loaded Set 11: experiment_run_11
Loaded Set 12: experiment_run_12
Loaded Set 13: experiment_run_13


### 1d. Large title fonts reserve more space automatically

`set_font_sizes(suptitle=...)` is applied *after* the base layout is built, so the
top margin is re-reserved for the real font size — a big title still clears the
legend and isn't clipped at the top edge.

In [7]:
nb.set_font_sizes(suptitle=34)
fig = nb.plot(
    x='time', y=['signal', 'temp'],
    suptitle='Large 34px Title<br>Space is reserved for the bigger font',
    figsize=(11, 6),
)
nb.set_font_sizes(reset=True)   # don't leak the big font into later cells
fig

### 1e. Before / after, side by side

Two hand-built Plotly figures with the **same** 3-line title and an 18-entry
horizontal legend. The first uses the *old* placement (legend bottom-anchored at
`y=1.02`, growing up); the second uses unichart's current `_base_layout`.

In [8]:
title = 'Three Line Title<br>Second Line<br>Third Line'
traces = [go.Scatter(x=[0, 1], y=[i, i + 1], name=f'series_{i:02d}') for i in range(18)]

# OLD: bottom-anchored horizontal legend in paper coords -> grows UP into the title
old = go.Figure(traces)
old.update_layout(
    template='plotly_white', width=820, height=520,
    title=dict(text=title, x=0.5, xanchor='center', y=0.99, yanchor='top', yref='container'),
    margin=dict(t=100),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
)

# NEW: unichart's centralized layout -> title pinned on top, legend below it
new = go.Figure(traces)
new.update_layout(**uc._base_layout(
    False, None, (8.2, 5.2),
    title={'text': title}, showlegend=True, legend=dict(orientation='h'),
))

print('BEFORE — legend overlaps the multi-line title:')
old.show()
print('AFTER — title protected, legend grows downward:')
new.show()

BEFORE — legend overlaps the multi-line title:


AFTER — title protected, legend grows downward:


---
## 2. `set_plot_size`

`figsize` fixes the size of the **whole figure**; the margins (title, legend,
colorbar) then eat into it, so the actual plot area changes from plot to plot.
`set_plot_size(width, height)` inverts that: it pins the **inner plot area** and
grows the figure to `plot area + margins`.

* Units are inches, same as `figsize`.
* Pass only the dimension you want to pin; `None` leaves it driven by `figsize`.
* `set_plot_size(reset=True)` (or both `None`) clears it.

### 2a. The problem — without it, the plot area shrinks as the title grows

Same `figsize=(8, 5)`, but a taller title steals vertical space from the plot.

In [9]:
nb.set_plot_size(reset=True)   # ensure the feature is OFF

for t in ['Short title',
          'Two line title<br>second line',
          'Four line title<br>line two<br>line three<br>line four']:
    fig = nb.plot(x='time', y='signal', suptitle=t, figsize=(8, 5))
    print(f'{t.count(chr(60)+"br"+chr(62)) + 1}-line title  ->  inner plot area = {paper_region(fig)} px')

1-line title  ->  inner plot area = (640, 331.9) px
2-line title  ->  inner plot area = (640, 305.8) px
4-line title  ->  inner plot area = (640, 253.6) px


### 2b. The fix — pin the inner area, the figure grows instead

Now the plot area is a constant **600×400 px** while the *figure* height grows to
absorb the taller titles.

In [10]:
nb.set_plot_size(width=6, height=4)   # inner area = 600 x 400 px

for t in ['Short title',
          'Two line title<br>second line',
          'Four line title<br>line two<br>line three<br>line four']:
    fig = nb.plot(x='time', y='signal', suptitle=t, figsize=(8, 5))
    print(f'plot area = {paper_region(fig)} px   '
          f'(figure = {int(fig.layout.width)} x {int(round(fig.layout.height))} px)')

plot area = (600, 400.0) px   (figure = 760 x 568 px)
plot area = (600, 400.0) px   (figure = 760 x 594 px)


plot area = (600, 400.0) px   (figure = 760 x 646 px)


Visually — these two plots have **identical** drawing areas despite very
different titles and figure heights:

In [11]:
nb.set_plot_size(width=6, height=4)
nb.plot(x='time', y='signal', suptitle='One line', figsize=(8, 5)).show()
nb.plot(x='time', y='signal',
        suptitle='Four lines<br>two<br>three<br>four', figsize=(8, 5)).show()

### 2c. Pin only one dimension

Pass just `height` (or just `width`); the other follows `figsize`.

In [12]:
nb.set_plot_size(height=4)   # height pinned to 400 px, width from figsize
for fs in [(6, 8), (10, 8)]:
    fig = nb.plot(x='time', y='signal', suptitle='Pinned height', figsize=fs)
    print(f'figsize={fs}  ->  plot area = {paper_region(fig)} px')

figsize=(6, 8)  ->  plot area = (440, 400.0) px


figsize=(10, 8)  ->  plot area = (840, 400.0) px


### 2d. Same plot area across different plot types

Bars, boxes, histograms and even a contour (which adds a colorbar) all land on
the same inner size.

> Note: each figure is measured **immediately** after it is built. Unichart frees
> the previous figure's data on the next plot call (a memory optimization), so
> don't stash figures and measure them later — measure as you go.

In [13]:
nb.set_plot_size(width=6, height=4)

builders = [
    ('plot',    lambda: nb.plot(x='time', y='signal', suptitle='plot', figsize=(9, 6))),
    ('bar',     lambda: nb.bar(x='group', y='signal', suptitle='bar', figsize=(9, 6))),
    ('box',     lambda: nb.box(x='group', y='signal', suptitle='box', figsize=(9, 6))),
    ('contour', lambda: nb.contour(x='time', y='temp', z='power', suptitle='contour', figsize=(9, 6))),
]
for name, build in builders:
    print(f'{name:8s} plot area = {paper_region(build())} px')   # build + measure together

plot     plot area = (600, 400.0) px


bar      plot area = (600, 400.0) px
box      plot area = (600, 400.0) px


contour  plot area = (600, 400.0) px


### 2e. Reset

Back to `figsize`-driven sizing.

In [14]:
nb.set_plot_size(reset=True)
fig = nb.plot(x='time', y='signal', suptitle='back to normal', figsize=(8, 5))
print('plot_size cleared. figure =', int(fig.layout.width), 'x', int(round(fig.layout.height)),
      '| plot area =', paper_region(fig))

plot_size cleared. figure = 800 x 500 | plot area = (640, 331.9)


### 2f. Caveat: colorbars / very long tick labels

The arithmetic is exact while Plotly's margin **autoexpand** stays inert — which
it does for the suptitle and the downward-growing legend. A colorbar (contour /
hue) or very long tick labels can overflow the reserved right/left margin and
trigger autoexpand, making *that one dimension* come out slightly short. In the
common case (the colorbar fits the reserved right margin) it matches exactly, as
in 2d above. This is documented in `set_plot_size`'s docstring.

In [15]:
print(uc.UnichartNotebook.set_plot_size.__doc__)

Pin the inner plot-area size so plots stay the same size regardless of
        suptitle lines, legend rows, or other margin changes.

        ``width``/``height`` are in inches (same units as ``figsize``). The figure
        is grown to ``plot_area + margins``, so the drawing region is held constant
        while margins absorb the title/legend. Pass only the dimension(s) you want
        to pin — ``None`` leaves that dimension driven by ``figsize``. Each call
        replaces the previous setting (calling with only ``height`` drops a prior
        ``width`` pin). ``reset=True`` (or both ``None``) clears it.

        Note: the arithmetic is exact when Plotly's margin ``autoexpand`` stays
        inert, which it does for the suptitle and the (downward-growing) legend.
        A colorbar (contour/hue) or very long tick labels can still autoexpand the
        right/left margin and make that dimension come out slightly short.
        


---
## 3. `footer` — a text box at the bottom of the figure

`footer` mirrors `suptitle`, but pins a (possibly multi-line) text box to the
**bottom** of the figure — handy for source notes, captions, or run metadata.

* Pass it per call: `nb.plot(..., footer='...')`
* Or set a persistent default: `nb.footer = '...'` (a per-call `footer=` wins)
* Size it via `set_font_sizes(footer=...)`
* If no footer is given, behavior is exactly as before (no extra bottom margin).

It is placed **below the x-axis labels** (the bottom margin grows to fit it) and
centered, mirroring the centered suptitle on top.

### 3a. A per-call footer

In [16]:
nb.set_plot_size(reset=True)
nb.footer = None
nb.plot(
    x='time', y=['signal', 'temp'],
    suptitle='Sensor Overview',
    footer='Source: synthetic data — generated for this demo   |   Figure 1',
    figsize=(11, 6),
)

### 3b. A persistent footer via the attribute

Set `nb.footer` once and it applies to every subsequent plot (until cleared),
just like `nb.suptitle`.

In [17]:
nb.footer = 'Confidential — internal use only'
fig = nb.bar(x='group', y='signal', suptitle='Quarterly', figsize=(11, 6))
nb.footer = None   # clear so later cells aren't affected
fig

### 3c. Multi-line footer + custom footer font size

`\n` (or `<br>`) makes multiple lines, and the bottom margin grows to fit them.

In [18]:
nb.set_font_sizes(footer=15)
fig = nb.plot(
    x='time', y='signal',
    suptitle='Footnotes',
    footer='Line one of the footnote\nLine two with more detail\nLine three: a reference',
    figsize=(11, 6),
)
nb.set_font_sizes(reset=True)
fig

### 3d. Everything together — suptitle + wrapping legend + footer

The multi-line title, the downward-growing legend, and the footer all coexist
without overlapping.

In [19]:
nb_many.plot(
    x='time', y='signal',
    suptitle=('Full Layout Demo'
              '<br>multi-line title on top, 14-series legend below it,'
              '<br>footer pinned to the bottom'),
    footer='Source: 14 synthetic experiment runs   |   units: arbitrary',
    figsize=(11, 7),
)

### 3e. Footer + `set_plot_size` — inner area stays constant

The reserved footer band is absorbed by the figure growing, so the plot area is
unchanged whether or not a footer is present (measured inline, as before):

In [20]:
nb.set_plot_size(width=6, height=4)   # inner area = 600 x 400 px
for label, kwargs in [('no footer',     {}),
                      ('1-line footer', {'footer': 'A footer'}),
                      ('3-line footer', {'footer': 'one\ntwo\nthree'})]:
    fig = nb.plot(x='time', y='signal', suptitle='t', figsize=(8, 5), **kwargs)
    print(f'{label:14s} -> plot area = {paper_region(fig)} px   '
          f'(figure height = {int(round(fig.layout.height))} px)')
nb.set_plot_size(reset=True)

no footer      -> plot area = (600, 400.0) px   (figure height = 568 px)
1-line footer  -> plot area = (600, 400.0) px   (figure height = 606 px)
3-line footer  -> plot area = (600, 400.0) px   (figure height = 640 px)
